In [0]:
import requests, json, uuid
from datetime import datetime, timezone, timedelta
from pyspark.sql import Row
from pyspark.sql.functions import col

# Databricks widget for job/manual parameter
dbutils.widgets.text("target_date", "")
target_date = dbutils.widgets.get("target_date").strip()

# Default to yesterday if no date is provided
if not target_date:
    target_date = (datetime.now(timezone.utc) - timedelta(days=1)).strftime("%Y-%m-%d")

RUN_ID = str(uuid.uuid4())
INGESTED_AT = datetime.now(timezone.utc)

AIRPORTS = ["AMS", "LHR", "JFK"]
WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"

# Read airport coordinates from bronze_airports_raw
airports_df = (
    spark.table("dev_project.flight_delay_lakehouse.bronze_airports_raw")
    .select("iata_code", "latitude_deg", "longitude_deg")
    .where(col("iata_code").isin(AIRPORTS))
)

airport_rows = airports_df.collect()

rows = []

for a in airport_rows:
    airport = a["iata_code"]
    lat = float(a["latitude_deg"])
    lon = float(a["longitude_deg"])

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": target_date,
        "end_date": target_date,
        "hourly": "temperature_2m,precipitation,wind_speed_10m",
        "timezone": "UTC"
    }

    r = requests.get(WEATHER_URL, params=params, timeout=30)
    r.raise_for_status()
    payload = r.json()

    rows.append(
        Row(
            run_id=RUN_ID,
            ingested_at=INGESTED_AT,
            source="open-meteo",
            airport_iata=airport,
            latitude=lat,
            longitude=lon,
            weather_kind="historical",
            query_params=json.dumps(params),
            raw_response_json=json.dumps(payload)
        )
    )

weather_historical_df = spark.createDataFrame(rows)

(
    weather_historical_df.write
        .format("delta")
        .mode("append")
        .saveAsTable("dev_project.flight_delay_lakehouse.bronze_weather_raw")
)